# Chapter 10 — Interpreting What ConvNets Learn

Maps to Chollet Ch.10. ConvNets aren't black boxes — their representations are *visual*. Three techniques:
1. **Intermediate activations** — what each layer "sees" for a given image.
2. **Filter visualization (gradient ascent)** — the pattern each filter maximally responds to.
3. **Grad-CAM heatmaps** — *which pixels* drove a particular class decision (the most practically useful).

We'll train a small ConvNet on CIFAR-10 and interpret it (no giant pretrained downloads).

In [ ]:
import os; os.environ["KERAS_BACKEND"]="tensorflow"
import keras, numpy as np, matplotlib.pyplot as plt, tensorflow as tf
from keras import layers

CLASSES = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]
(Xtr, ytr), (Xte, yte) = keras.datasets.cifar10.load_data(); ytr, yte = ytr[:,0], yte[:,0]
Xtr, ytr = Xtr[:10000]/255., ytr[:10000]; Xte, yte = Xte[:1000]/255., yte[:1000]

inp = keras.Input((32,32,3))
x = layers.Conv2D(32, 3, activation="relu", padding="same", name="conv1")(inp); x = layers.MaxPooling2D()(x)
x = layers.Conv2D(64, 3, activation="relu", padding="same", name="conv2")(x); x = layers.MaxPooling2D()(x)
x = layers.Conv2D(64, 3, activation="relu", padding="same", name="last_conv")(x)
g = layers.GlobalAveragePooling2D()(x)
out = layers.Dense(10, activation="softmax")(g)
model = keras.Model(inp, out)
model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(Xtr, ytr, epochs=8, batch_size=64, verbose=0)
print("small CNN test acc:", round(model.evaluate(Xte, yte, verbose=0, return_dict=True)["accuracy"], 3),
      "(low budget; fine for visualization)")


## 1. Visualizing intermediate activations
Build a **multi-output model** (Ch.7) that returns every Conv layer's output for one image. Early layers keep
edge/texture detail; deeper layers get sparser and more abstract — the *information-distillation pipeline*.

In [ ]:
conv_layers = [l for l in model.layers if isinstance(l, layers.Conv2D)]
act_model = keras.Model(model.input, [l.output for l in conv_layers])

img = Xte[3:4]                                        # one test image
activations = act_model.predict(img, verbose=0)
print("layer activation shapes:", [a.shape for a in activations])

# show input + first 8 channels of the first conv layer
fig, ax = plt.subplots(1, 9, figsize=(14, 2))
ax[0].imshow(img[0]); ax[0].set_title("input"); ax[0].axis("off")
for i in range(8):
    ax[i+1].imshow(activations[0][0,:,:,i], cmap="viridis"); ax[i+1].axis("off")
    ax[i+1].set_title(f"conv1 ch{i}", fontsize=8)
plt.suptitle("first-layer feature maps = edge/texture detectors"); plt.tight_layout(); plt.show()


## 2. Visualizing filters via gradient ascent
What input maximizes a given filter? Start from noise and do **gradient *ascent*** on the input pixels to
maximize that filter's mean activation. (This is the Ch.2 gradient loop, but we optimize the *image*, not the
weights.) On small nets the patterns look noisy; on big pretrained nets they reveal edges→textures→object parts.

In [ ]:
feature_extractor = keras.Model(model.input, model.get_layer("conv2").output)

@tf.function
def ascent_step(image, filter_index, lr=10.0):
    with tf.GradientTape() as tape:
        tape.watch(image)                              # image isn't a Variable -> watch it
        activation = feature_extractor(image)
        loss = tf.reduce_mean(activation[:, 2:-2, 2:-2, filter_index])  # ignore borders
    grads = tape.gradient(loss, image)
    grads = grads / (tf.sqrt(tf.reduce_mean(tf.square(grads))) + 1e-8)  # normalize trick
    return image + lr * grads

def visualize_filter(filter_index, steps=30):
    image = tf.random.uniform((1,32,32,3), 0.4, 0.6)
    for _ in range(steps):
        image = ascent_step(image, filter_index)
    image = image[0].numpy()
    image -= image.mean(); image /= (image.std()+1e-8); image = image*0.25 + 0.5
    return np.clip(image, 0, 1)

fig, ax = plt.subplots(1, 6, figsize=(12, 2.2))
for i in range(6):
    ax[i].imshow(visualize_filter(i)); ax[i].axis("off"); ax[i].set_title(f"filter {i}", fontsize=8)
plt.suptitle("patterns that maximally activate conv2 filters"); plt.tight_layout(); plt.show()


## 3. Grad-CAM — which pixels drove the decision (the key technique)
**Recipe:** take the last conv layer's feature maps for an image; compute the gradient of the predicted class
score w.r.t. those maps; **global-average-pool** the gradients → one weight per channel; weight-sum the
feature maps; ReLU + normalize → a coarse heatmap; upscale and overlay on the image. Red marks the region
that made the model predict that class.

In [ ]:
def grad_cam(image, class_index=None):
    grad_model = keras.Model(model.input, [model.get_layer("last_conv").output, model.output])
    img = tf.convert_to_tensor(image[None])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img)
        if class_index is None: class_index = int(tf.argmax(preds[0]))
        class_score = preds[:, class_index]
    grads  = tape.gradient(class_score, conv_out)         # (1,8,8,64)
    weights = tf.reduce_mean(grads, axis=(0,1,2))         # (64,) channel importances
    heatmap = tf.reduce_sum(conv_out[0] * weights, axis=-1)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), class_index

def show_gradcam(idx):
    image = Xte[idx]
    heat, cls = grad_cam(image)
    heat_big = np.array(keras.ops.image.resize(heat[...,None], (32,32)))[:,:,0]
    fig, ax = plt.subplots(1, 3, figsize=(8, 2.6))
    ax[0].imshow(image); ax[0].set_title(f"input (true={CLASSES[yte[idx]]})"); ax[0].axis("off")
    ax[1].imshow(heat, cmap="jet"); ax[1].set_title("Grad-CAM (8x8)"); ax[1].axis("off")
    ax[2].imshow(image); ax[2].imshow(heat_big, cmap="jet", alpha=0.5)
    ax[2].set_title(f"overlay (pred={CLASSES[cls]})"); ax[2].axis("off")
    plt.tight_layout(); plt.show()

for idx in [0, 3, 12]:
    show_gradcam(idx)


**Reading Grad-CAM:** if the hot region sits on the object → the model is "looking" at the right thing.
If it's on the background, the model may be using a spurious correlation (Ch.5) — a red flag even when
accuracy looks fine. This is your go-to tool for *debugging* misclassifications and building trust.

---
# ✍️ PROBLEMS

### P1 — Activation sparsity
Quantify the claim "deeper layers are sparser": for each conv layer, compute the fraction of activation
values that are ~0 for a test image, averaged over 50 images. Plot fraction-zero vs layer depth.

In [ ]:
# TODO


### P2 — Grad-CAM on mistakes
Find 5 test images the model **misclassifies**. Run Grad-CAM for both the true and predicted class. Where is
the model looking? Write one sentence per image explaining the error.

In [ ]:
# TODO


### P3 — Class contrast
Pick one image and produce Grad-CAM heatmaps for several different `class_index` values (not just the
predicted one). Do different classes highlight different regions? What does that tell you?

In [ ]:
# TODO


### P4 — Filter viz on a pretrained net (Colab)
Load `keras.applications.Xception(weights="imagenet", include_top=False)`. Run the gradient-ascent filter
visualizer on an early vs a deep separable-conv layer. Confirm early=edges/colors, deep=textures/parts.

In [ ]:
# TODO


---
# 📋 TEMPLATES

### T1 — Intermediate activations (multi-output model)

In [ ]:
act_model = keras.Model(model.input,
                        [l.output for l in model.layers if isinstance(l, layers.Conv2D)])
activations = act_model.predict(image[None])      # list, one array per conv layer
# plt.imshow(activations[layer][0, :, :, channel], cmap="viridis")


### T2 — Filter visualization via gradient ascent (TF)

In [ ]:
import tensorflow as tf
fe = keras.Model(model.input, model.get_layer(LAYER).output)
@tf.function
def ascent_step(image, fidx, lr=10.0):
    with tf.GradientTape() as tape:
        tape.watch(image); loss = tf.reduce_mean(fe(image)[:, 2:-2, 2:-2, fidx])
    g = tape.gradient(loss, image); g = g/(tf.sqrt(tf.reduce_mean(tf.square(g)))+1e-8)
    return image + lr*g
img = tf.random.uniform((1,H,W,3), 0.4, 0.6)
for _ in range(30): img = ascent_step(img, FILTER_INDEX)


### T3 — Grad-CAM (drop-in)

In [ ]:
import tensorflow as tf, numpy as np, keras
def grad_cam(model, image, last_conv_name, class_index=None):
    gm = keras.Model(model.input, [model.get_layer(last_conv_name).output, model.output])
    img = tf.convert_to_tensor(image[None])
    with tf.GradientTape() as tape:
        conv_out, preds = gm(img)
        if class_index is None: class_index = int(tf.argmax(preds[0]))
        score = preds[:, class_index]
    grads = tape.gradient(score, conv_out)
    weights = tf.reduce_mean(grads, axis=(0,1,2))
    heat = tf.reduce_sum(conv_out[0]*weights, axis=-1)
    heat = tf.maximum(heat,0)/(tf.reduce_max(heat)+1e-8)
    return heat.numpy(), class_index
# overlay: resize heat to image size, imshow(image); imshow(heat, cmap="jet", alpha=0.5)


---
### ✅ Checklist
- [ ] Build a multi-output model to extract intermediate activations; explain edges→abstract progression.
- [ ] Run gradient *ascent* on the input to visualize what a filter responds to (and why patterns differ by depth).
- [ ] Implement Grad-CAM from scratch and overlay it on an image.
- [ ] Use Grad-CAM to debug misclassifications / detect spurious focus.

**Next: Chapter 11** — *Image segmentation* (predict a class per pixel; encoder–decoder / U-Net). Say "Chapter 11".